# 🏪 Laboratorio integrador: del dato a la decisión con el caso Mayoristas

## 📖 La historia

Un **distribuidor mayorista de alimentos en Portugal** vende a 440 clientes: hoteles, restaurantes y cafés (el canal **Horeca**) y tiendas de autoservicio y abarrotes (el canal **Retail**). Hoy los atiende a todos igual: el mismo vendedor, el mismo catálogo y el mismo camión.

El director comercial sospecha que eso le cuesta dinero: un restaurante que compra pescado fresco todos los días no necesita lo mismo que una tienda que compra cajas de detergente cada mes. Te entrega una tabla con el **gasto anual de cada cliente en seis categorías** y te pide una propuesta.

Este es el **cierre del curso**: vas a usar, en un solo proyecto, todo lo que aprendiste. Transformar y escalar (sesión 0), reducir con PCA (sesión 3), agrupar con K-Means (sesión 1), detectar atípicos con DBSCAN (sesión 2) y, sobre todo, **traducir los clusters a decisiones**.

> ### ❓ ¿Cómo debe organizar el distribuidor a sus clientes mayoristas: qué equipos de venta, qué surtido y qué logística le tocan a cada grupo?

## 🧭 Lo que vas a aprender

1. A reconocer datos de gasto **muy sesgados** y por qué conviene aplicar `log1p` **antes** de escalar.
2. A usar **PCA** como paso intermedio: resumir 6 categorías en 2 ejes que se pueden interpretar y graficar.
3. A combinar **K-Means** (segmentos) y **DBSCAN** (clientes atípicos) sobre el mismo espacio.
4. A **validar** un agrupamiento con una etiqueta que **no** usaste para agrupar (el canal de venta).
5. A convertir cada segmento en una **recomendación de negocio** con números en las unidades originales.

## ⏱️ Agenda (3 horas)

| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación, recap del curso y carga de datos | 10 min |
| 1 | Exploración: distribuciones sesgadas y outliers enormes | 20 min |
| 2 | ¿Por qué `log1p` antes de escalar? | 20 min |
| 3 | PCA: de 6 categorías a 2 ejes con significado | 25 min |
| ☕ | Descanso | 10 min |
| 4 | K-Means sobre los componentes principales | 25 min |
| 5 | DBSCAN: ¿quién no encaja en ningún segmento? | 20 min |
| 6 | Perfil de los segmentos en unidades originales | 20 min |
| 7 | Validación contra el canal de venta | 15 min |
| 8 | Recomendaciones de negocio y cierre | 15 min |
| | **Total** | **180 min** |

## 📋 Las variables

Datos: *Wholesale customers* (UCI Machine Learning Repository, Cardoso, 2014; licencia CC BY 4.0). Los gastos están en **unidades monetarias anuales** (u.m.).

| Variable (original) | En el notebook | Descripción | Unidades |
|---|---|---|---|
| `Channel` | `canal` | Horeca (hoteles, restaurantes, cafés) o Retail | categoría |
| `Region` | `region` | Lisboa, Oporto u Otra | categoría |
| `Fresh` | `frescos` | Gasto anual en productos frescos (frutas, verduras, carne, pescado) | u.m./año |
| `Milk` | `lacteos` | Gasto anual en lácteos | u.m./año |
| `Grocery` | `abarrotes` | Gasto anual en abarrotes (productos secos de anaquel) | u.m./año |
| `Frozen` | `congelados` | Gasto anual en congelados | u.m./año |
| `Detergents_Paper` | `detergentes_papel` | Gasto anual en detergentes y productos de papel | u.m./año |
| `Delicassen` | `delicatessen` | Gasto anual en delicatessen (embutidos, quesos finos) | u.m./año |

> ⚠️ **Regla del caso:** `canal` y `region` **no** entran al agrupamiento. Sólo agrupamos con los seis gastos. El canal lo guardamos para el final, como "examen" de nuestros segmentos.

---
# 0️⃣ Preparación (10 min)

## 📋 Recap: lo que ya sabes de las sesiones 1 a 3

| Sesión | Técnica | La idea clave | La trampa que aprendiste a evitar |
|---|---|---|---|
| 1 | **K-Means** | Asigna cada punto al centroide más cercano y recalcula; tú eliges `k` | Elegir `k` sólo con el codo; olvidar `n_init` y `random_state` |
| 2 | **DBSCAN** | Un cluster es una zona densa; lo que no cabe en ninguna es **ruido** (-1) | Elegir `eps` al azar en vez de usar la gráfica de k-distancia |
| 3 | **PCA** | Rota los datos para que pocos ejes (componentes) concentren la varianza | Aplicarlo sin escalar; no leer las cargas |
| Todas | **Escalado** | Las distancias y la varianza dependen de las unidades | Agrupar con variables en escalas distintas |

Hoy los encadenamos: **preparar → reducir → agrupar → validar → interpretar → actuar**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [2]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Cargar los datos

El archivo viene en la carpeta `datos/` junto a este notebook. Renombramos las columnas al español y traducimos los códigos de `canal` y `region` para leerlos más fácil.

In [3]:
# Esta celda va de regalo: léela con calma y ejecútala
import pandas as pd
import numpy as np

df = pd.read_csv('datos/wholesale_customers.csv')

df = df.rename(columns={
    'Channel': 'canal',
    'Region': 'region',
    'Fresh': 'frescos',
    'Milk': 'lacteos',
    'Grocery': 'abarrotes',
    'Frozen': 'congelados',
    'Detergents_Paper': 'detergentes_papel',
    'Delicassen': 'delicatessen',
})
df['canal'] = df['canal'].map({1: 'Horeca', 2: 'Retail'})
df['region'] = df['region'].map({1: 'Lisboa', 2: 'Oporto', 3: 'Otra'})

# Las seis variables de gasto: son las ÚNICAS que entran al agrupamiento
gasto = ['frescos', 'lacteos', 'abarrotes', 'congelados', 'detergentes_papel', 'delicatessen']

df.head()

,canal,region,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen
0,Retail,Otra,12669,9656,7561,214,2674,1338
1,Retail,Otra,7057,9810,9568,1762,3293,1776
2,Retail,Otra,6353,8808,7684,2405,3516,7844
3,Horeca,Otra,13265,1196,4221,6404,507,1788
4,Retail,Otra,22615,5410,7198,3915,1777,5185


---
# 1️⃣ Exploración: distribuciones sesgadas y outliers enormes (20 min)

Antes de agrupar hay que conocer los datos: cuántos son, si hay nulos, en qué rango se mueve cada gasto y cómo se relacionan entre sí.

In [4]:
# ¿Cuántas filas y columnas? ¿Hay valores nulos?
# Pista: df.shape y df.isnull().sum()
print(df.shape)
df.isnull().sum()


(440, 8)


canal                0
region               0
frescos              0
lacteos              0
abarrotes            0
congelados           0
detergentes_papel    0
delicatessen         0
dtype: int64

In [5]:
# Estadísticas descriptivas de los gastos
# Pista: df[gasto].describe().round(0)
df[gasto].describe().round(0)


,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen
count,440.0,440.0,440.0,440.0,440.0,440.0
mean,12000.0,5796.0,7951.0,3072.0,2881.0,1525.0
std,12647.0,7380.0,9503.0,4855.0,4768.0,2820.0
min,3.0,55.0,3.0,25.0,3.0,3.0
25%,3128.0,1533.0,2153.0,742.0,257.0,408.0
50%,8504.0,3627.0,4756.0,1526.0,816.0,966.0
75%,16934.0,7190.0,10656.0,3554.0,3922.0,1820.0
max,112151.0,73498.0,92780.0,60869.0,40827.0,47943.0


In [6]:
# ¿Cuántos clientes hay por canal y por región?
# Pista: pd.crosstab(df['region'], df['canal'], margins=True)
pd.crosstab(df['region'], df['canal'], margins=True)


canal,Horeca,Retail,All
region,,,
Lisboa,59,18,77
Oporto,28,19,47
Otra,211,105,316
All,298,142,440


✍️ **¿Qué observas?** ¿Hay nulos? Compara la media y la mediana (50%) de cada gasto: ¿qué te dice que la media sea mucho mayor? ¿Cuánto vale el máximo comparado con la mediana? ¿Qué canal domina?

- **Nulos:** no hay. Son 440 filas y 8 columnas, y todas tienen 0 nulos.
- **Media vs. mediana:** la media es mucho mayor que la mediana en todos los gastos (frescos 12,000 vs 8,504; detergentes_papel 2,881 vs 816). Eso indica una cola larga a la derecha: unos pocos clientes enormes jalan el promedio.
- **Máximo vs. mediana:** el máximo vale desde 13 veces la mediana (frescos) hasta 50 veces (detergentes_papel y delicatessen).
- **Canal:** domina Horeca con 298 clientes (68 %) contra 142 de Retail (32 %). Por región, 316 de 440 clientes están en 'Otra'.

In [7]:
# Histograma + boxplot de cada gasto
# Pista: plot_distributions(df, gasto)
plot_distributions(df, gasto)


In [8]:
# Esta celda va de regalo: léela con calma y ejecútala
# Dos números que resumen el sesgo de cada variable
sesgo = pd.DataFrame({
    'asimetria': df[gasto].skew().round(2),          # 0 = simétrica; > 1 = muy sesgada a la derecha
    'max_entre_mediana': (df[gasto].max() / df[gasto].median()).round(1),
})
sesgo

,asimetria,max_entre_mediana
frescos,2.56,13.2
lacteos,4.05,20.3
abarrotes,3.59,19.5
congelados,5.91,39.9
detergentes_papel,3.63,50.0
delicatessen,11.15,49.7


✍️ **¿Qué observas?** ¿Qué forma tienen los histogramas? ¿Qué variable es la más sesgada? ¿Cuántas veces la mediana gasta el cliente más grande? Si calculáramos distancias con estos datos, ¿qué clientes dominarían?

- **Forma:** los histogramas están muy sesgados a la derecha: una barra alta cerca de 0 y una cola larga con pocos clientes.
- **Más sesgada:** delicatessen (asimetría 11.15), seguida de congelados (5.91).
- **Máximo vs. mediana:** el cliente más grande gasta entre 13 veces (frescos) y 50 veces (detergentes_papel) la mediana.
- **Distancias:** dominarían esos pocos clientes enormes. Un solo cliente a 16 desviaciones estándar pesa más que cientos de clientes típicos.

In [9]:
# Matriz de correlación entre los gastos
# Pista: plot_correlation_matrix(df, gasto)
plot_correlation_matrix(df, gasto)


✍️ **¿Qué observas?** ¿Qué pares de variables están más correlacionados? ¿Ves un "bloque" de variables que suben juntas? ¿Qué tipo de negocio compraría esa canasta? ¿Por qué esto anticipa que PCA servirá?

- **Pares más correlacionados:** abarrotes–detergentes_papel (r ≈ 0.92), abarrotes–lacteos (≈ 0.73) y lacteos–detergentes_papel (≈ 0.66).
- **Bloque:** lacteos, abarrotes y detergentes_papel suben juntos. Es la canasta de una tienda de autoservicio que revende productos de despensa y limpieza (Retail).
- **Otro grupo:** frescos y congelados van más por su lado (r ≈ 0.35 entre ellos y casi nula con el bloque anterior).
- **Por qué PCA servirá:** seis variables se pueden resumir en pocos ejes, por ejemplo un eje de despensa/limpieza y otro de perecederos.

---
# 2️⃣ ¿Por qué `log1p` antes de escalar? (20 min)

`StandardScaler` resta la media y divide entre la desviación estándar, pero **no cambia la forma** de la distribución: un cliente que estaba 16 desviaciones arriba sigue igual de lejos. Para datos de dinero, el truco clásico es aplicar el **logaritmo** antes de escalar:

$$x' = \log(1 + x)$$

- El logaritmo **comprime la cola**: pasar de 1,000 a 10,000 pesa lo mismo que pasar de 10,000 a 100,000 (multiplicar por 10).
- Usamos `log1p` (es decir, $\log(1+x)$) en vez de `log` para que un gasto de 0 no dé $-\infty$.
- En lenguaje de negocio: comparamos **proporciones** ("gasta el doble") en vez de diferencias absolutas ("gasta 5,000 más").

Vamos a escalar **de las dos formas** y comparar.

In [10]:
# 1. Crea df_log aplicando np.log1p a df[gasto] (el df original NO se modifica)
# 2. Grafica sus distribuciones con plot_distributions(df_log, gasto)
df_log = np.log1p(df[gasto])
plot_distributions(df_log, gasto)


In [11]:
# Esta celda va de regalo: léela con calma y ejecútala
# Asimetría antes y después del logaritmo
pd.DataFrame({
    'asimetria_original': df[gasto].skew().round(2),
    'asimetria_log': df_log.skew().round(2),
})

,asimetria_original,asimetria_log
frescos,2.56,-1.58
lacteos,4.05,-0.22
abarrotes,3.59,-0.67
congelados,5.91,-0.35
detergentes_papel,3.63,-0.24
delicatessen,11.15,-1.09


✍️ **¿Qué observas?** ¿Cómo cambió la forma de los histogramas? Compara la asimetría antes y después. ¿Alguna variable quedó sesgada hacia el otro lado? ¿Qué clientes crees que están ahora en esa cola izquierda?

- **Forma:** después de `log1p` los histogramas se ven mucho más simétricos, con forma de campana. La cola derecha se comprimió.
- **Asimetría:** bajó en valor absoluto en todas (delicatessen 11.15 → −1.09, congelados 5.91 → −0.35, lacteos 4.05 → −0.22).
- **Al otro lado:** frescos (−1.58), delicatessen (−1.09) y abarrotes (−0.67) quedaron sesgadas hacia la izquierda.
- **Cola izquierda:** son clientes que casi no compran esa categoría (gastos cercanos a 0, por ejemplo 3 u 85). Con montos crudos se confundían con los demás clientes pequeños y con log se separan.

In [12]:
from sklearn.preprocessing import StandardScaler

# 1. scaler_crudo = StandardScaler();  X_crudo = scaler_crudo.fit_transform(df[gasto])
# 2. scaler_log = StandardScaler();    X_log = scaler_log.fit_transform(df_log)
# 3. Imprime X_crudo.shape y X_log.shape
scaler_crudo = StandardScaler()
X_crudo = scaler_crudo.fit_transform(df[gasto])
scaler_log = StandardScaler()
X_log = scaler_log.fit_transform(df_log)
print(X_crudo.shape, X_log.shape)


(440, 6) (440, 6)


In [13]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Qué tan lejos queda el cliente más extremo de cada variable? (en desviaciones estándar)
pd.DataFrame({
    'max_|z|_crudo': np.abs(X_crudo).max(axis=0).round(1),
    'max_|z|_log': np.abs(X_log).max(axis=0).round(1),
    'celdas_|z|>3_crudo': (np.abs(X_crudo) > 3).sum(axis=0),
    'celdas_|z|>3_log': (np.abs(X_log) > 3).sum(axis=0),
}, index=gasto)

,max_|z|_crudo,max_|z|_log,celdas_|z|>3_crudo,celdas_|z|>3_log
frescos,7.9,5.0,7,6
lacteos,9.2,3.8,9,2
abarrotes,8.9,6.4,7,2
congelados,11.9,3.2,6,1
detergentes_papel,8.0,3.2,10,2
delicatessen,16.5,4.1,4,8


### 🧪 Prueba rápida: ¿cambia el agrupamiento?

Ajustamos un K-Means con k = 4 en cada espacio (un valor cualquiera, sólo para comparar; el k definitivo lo elegimos en la Parte 4) y vemos **cuántos clientes caen en cada cluster**.

In [14]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.cluster import KMeans

kmeans_crudo = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_crudo = kmeans_crudo.fit_predict(X_crudo)

print('Tamaños con datos crudos:', np.bincount(labels_crudo))
print('Silhouette (espacio crudo):', evaluar_clustering('crudo', X_crudo, labels_crudo)['silhouette'])

Tamaños con datos crudos: [110 315  10   5]
Silhouette (espacio crudo): 0.348


In [15]:
# Haz lo mismo con X_log:
# 1. kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
# 2. labels_log = kmeans_log.fit_predict(X_log)
# 3. Imprime los tamaños con np.bincount(labels_log)
# 4. Imprime la silhouette: evaluar_clustering('log', X_log, labels_log)['silhouette']
kmeans_log = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_log = kmeans_log.fit_predict(X_log)
print('Tamaños con datos log:', np.bincount(labels_log))
print('Silhouette (espacio log):', evaluar_clustering('log', X_log, labels_log)['silhouette'])


Tamaños con datos log:

 [ 61 129 132 118]
Silhouette (espacio log): 0.188


✍️ **¿Qué observas?** ¿Qué tan lejos queda el cliente más extremo en cada versión? Compara los tamaños de los clusters: ¿cuál te serviría para organizar equipos de venta? La silhouette del espacio crudo sale más alta: ¿qué está premiando? ¿Por qué no deberías compararla con la silhouette calculada en el espacio log?

- **Cliente más extremo:** queda a 16.5 desviaciones estándar (delicatessen) en el espacio crudo y a 6.4 como máximo en el espacio log.
- **Tamaños:** con datos crudos salen [110, 315, 10, 5]: dos clusters con 10 y 5 clientes son solo outliers y uno concentra a 315. Con log salen [61, 129, 132, 118], grupos parejos que sí sirven para organizar equipos de venta.
- **Silhouette cruda (0.348):** sale más alta porque premia que unos pocos clientes enormes queden aislados en clusters diminutos y que el resto se vea compacto. No es un buen agrupamiento para el negocio.
- **No compararla con la del log (0.188):** son espacios distintos, con otras escalas y otras distancias, así que los dos números no miden lo mismo.

---
# 3️⃣ PCA: de 6 categorías a 2 ejes con significado (25 min)

Vimos que varias categorías se mueven juntas. PCA busca las direcciones que concentran la mayor varianza; si dos o tres ejes resumen casi todo, podemos **agrupar y graficar** en ese espacio reducido. Primero ajustamos PCA con **todos** los componentes para decidir cuántos quedarnos.

In [16]:
from sklearn.decomposition import PCA

# 1. pca_log = PCA() y ajústalo con pca_log.fit(X_log)
# 2. Imprime pca_log.explained_variance_ratio_ y su acumulada con np.cumsum(...)
# 3. Grafica con plot_varianza_explicada(pca_log)
pca_log = PCA()
pca_log.fit(X_log)
print(pca_log.explained_variance_ratio_.round(3))
print(np.cumsum(pca_log.explained_variance_ratio_).round(3))
plot_varianza_explicada(pca_log)


[0.441 0.272 0.107 0.101 0.049 0.03 ]
[0.441 0.713 0.82  0.921 0.97  1.   ]


✍️ **¿Qué observas?** ¿Cuánta varianza explican PC1 y PC2 juntos? ¿Dónde ves el codo? ¿Cuántos componentes necesitarías para el 80%? ¿Con cuántos te quedarías y por qué?

- **PC1 + PC2:** explican 71.3 % de la varianza (44.1 % + 27.2 %).
- **Codo:** aparece en el tercer componente. De PC3 en adelante cada uno aporta 10 % o menos.
- **80 %:** se necesitan 3 componentes (82.0 % acumulado).
- **Mi elección:** me quedo con 2. Explican más del 70 %, permiten graficar y sus ejes se pueden nombrar. PC3 sumaría solo 10.7 %.

In [17]:
# ¿Qué variables pesan en cada componente?
# Pista: plot_cargas(pca_log, gasto, n_componentes=3)
plot_cargas(pca_log, gasto, n_componentes=3)


✍️ **¿Qué observas?** Completa la tabla con las variables que más pesan en cada componente y ponle un nombre de negocio a cada eje.

| Componente | Variables con más peso | Nombre de negocio |
|---|---|---|
| PC1 | abarrotes (0.57), detergentes_papel (0.55), lacteos (0.54) | Canasta de despensa y limpieza (tipo Retail) |
| PC2 | frescos (0.59), congelados (0.59), delicatessen (0.53) | Canasta de perecederos (tipo Horeca) |

PC1 reúne a la canasta de despensa y limpieza (lacteos, abarrotes y detergentes_papel pesan de forma parecida). PC2 reúne a los productos de un restaurante o café (frescos, congelados y delicatessen).

In [18]:
# 1. pca_final = PCA(n_components=<los componentes que elegiste>)
# 2. X_pcs = pca_final.fit_transform(X_log)
# 3. df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
# 4. Grafica con plot_biplot(X_pcs, pca_final, gasto)   (dibuja PC1 contra PC2)
pca_final = PCA(n_components=2)
X_pcs = pca_final.fit_transform(X_log)
df_pcs = pd.DataFrame(X_pcs, columns=[f'PC{i + 1}' for i in range(X_pcs.shape[1])], index=df.index)
plot_biplot(X_pcs, pca_final, gasto)


✍️ **¿Qué observas?** ¿Hacia dónde apuntan las flechas de cada canasta? ¿Ves grupos claramente separados en la nube o una sola masa? ¿Hay clientes lejos del resto? ¿Qué significa estar arriba a la derecha del biplot?

- **Flechas:** lacteos, abarrotes y detergentes_papel apuntan juntas hacia la derecha (PC1). Frescos, congelados y delicatessen apuntan hacia arriba (PC2).
- **Grupos:** no se ven grupos claramente separados. Es una sola masa continua, más densa cerca del centro.
- **Lejos del resto:** sí hay clientes alejados, sobre todo hacia la derecha y hacia arriba.
- **Arriba a la derecha:** significa gastar mucho en las dos canastas a la vez, es decir, un cliente grande en casi todo.

---
# ☕ Descanso (10 min)

---
# 4️⃣ K-Means sobre los componentes principales (25 min)

Ahora sí, agrupamos en el espacio `X_pcs` (los componentes que elegiste). Como en la sesión 1, elegimos `k` con **varias señales**: codo, silhouette y gráfica de silueta... y con **sentido de negocio**.

In [19]:
# Inercia y silhouette de K-Means para k = 2 ... 8 en el espacio de componentes
# Pista: tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
tabla_k = tabla_metricas_k(X_pcs, range(2, 9))
tabla_k


,k,inercia,silhouette
0,2,1086.1,0.408
1,3,796.6,0.392
2,4,632.8,0.329
3,5,531.7,0.340
4,6,444.0,0.352
5,7,379.4,0.362
6,8,333.3,0.357


In [20]:
# 1. Grafica el codo: plot_codo(tabla_k['k'], tabla_k['inercia'])
# 2. Grafica la silhouette: plot_silueta_k(tabla_k)
plot_codo(tabla_k['k'], tabla_k['inercia'])
plot_silueta_k(tabla_k)


✍️ **¿Qué observas?** ¿Dónde ves el codo? ¿Qué k prefiere la silhouette? ¿Coinciden? Si los candidatos casi empatan, ¿qué criterio usarías para decidir? Escribe el k que eliges y por qué.

- **Codo:** la inercia baja fuerte de k = 2 a k = 4 y después se aplana. No hay un codo muy nítido.
- **Silhouette:** la más alta es k = 2 (0.408) y k = 3 casi empata (0.392).
- **¿Coinciden?** No del todo: el codo sugiere 3 o 4 y la silhouette sugiere 2.
- **Criterio de desempate:** el negocio. k = 2 solo reproduciría Horeca contra Retail, que el distribuidor ya conoce.
- **Mi k:** elijo k = 4. Da segmentos accionables y de tamaño razonable (118, 106, 72 y 144), aunque su silhouette (0.329) sea menor.

In [21]:
# 1. kmeans_final = KMeans(n_clusters=<tu k>, n_init=10, random_state=42)
# 2. labels_final = kmeans_final.fit_predict(X_pcs)
# 3. Imprime los tamaños con np.bincount(labels_final)
# 4. plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_)
K_FINAL = 4
kmeans_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42)
labels_final = kmeans_final.fit_predict(X_pcs)
print('Tamaños:', np.bincount(labels_final))
plot_clusters(df_pcs, 'PC1', 'PC2', labels_final, centroides=kmeans_final.cluster_centers_)


Tamaños: [118 106  72 144]


In [22]:
# Gráfica de silueta de tu modelo
# Pista: plot_silueta(X_pcs, labels_final)
plot_silueta(X_pcs, labels_final)


In [23]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Perdimos algo por agrupar en los componentes principales y no en las 6 variables (log + escaladas)?
kmeans_6d = KMeans(n_clusters=kmeans_final.n_clusters, n_init=10, random_state=42)
labels_6d = kmeans_6d.fit_predict(X_log)

# Tabla cruzada: filas = grupo usando los PCs, columnas = grupo usando las 6 variables
# (los números de cluster son arbitrarios: el "0" de un modelo puede ser el "1" del otro)
tabla_pcs_vs_6d = pd.crosstab(
    pd.Series(labels_final, name='K-Means en PCs'),
    pd.Series(labels_6d, name='K-Means en 6 variables'),
)
# Para cada grupo de los PCs, la celda más grande son los clientes que "se quedan juntos"
print('Clientes que quedan en el mismo grupo:', tabla_pcs_vs_6d.max(axis=1).sum(), 'de', len(labels_final))
tabla_pcs_vs_6d

Clientes que quedan en el mismo grupo: 408 de 440


K-Means en 6 variables,0,1,2,3
K-Means en PCs,,,,
0,0,0,118,0
1,0,3,0,103
2,61,0,2,9
3,0,126,12,6


✍️ **¿Qué observas?** ¿Cuántos clientes quedaron en cada cluster? ¿Dónde está cada cluster en el plano PC1-PC2 y qué significa eso en términos de canastas? ¿La silueta es alta o moderada? ¿Por qué? ¿Perdimos información al agrupar con los componentes en vez de las 6 variables?

- **Tamaños:** 118, 106, 72 y 144 clientes.
- **Ubicación:** según sus perfiles, los clusters 1 y 2 están hacia la derecha (PC1 alto, canasta de despensa), el cluster 3 hacia arriba (PC2 alto, frescos y congelados) y el cluster 0 es el de menor gasto en todo.
- **Silueta:** es moderada (0.329), porque los clientes forman un continuo sin fronteras claras entre grupos.
- **¿Perdimos información?** Poca. Con 2 componentes en lugar de 6 variables, 408 de 440 clientes (93 %) quedan en el mismo grupo.

### 📊 Guardamos los resultados

In [24]:
# 1. comparacion = []
# 2. comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))
comparacion = []
comparacion.append(evaluar_clustering('K-Means final (PCs)', X_pcs, labels_final))


---
# 5️⃣ DBSCAN: ¿quién no encaja en ningún segmento? (20 min)

K-Means **obliga** a cada cliente a pertenecer a un cluster, aunque esté lejísimos de todos. DBSCAN no: lo que no está en una zona densa queda como **ruido**. Lo usamos sobre el **mismo espacio** `X_pcs` para encontrar clientes atípicos.

Receta de la sesión 2:
1. `min_samples` ≈ 2 × número de dimensiones. Calcúlalo con los componentes que elegiste: `min_samples = 2 * X_pcs.shape[1]`.
2. `eps` sale de la **gráfica de k-distancia** con k = `min_samples`: buscamos el codo donde la curva se dispara.

In [25]:
# 1. min_samples = 2 * X_pcs.shape[1]
# 2. Grafica la k-distancia: plot_k_distancia(X_pcs, k=min_samples)
# 3. Ubica el codo (donde la curva se dispara) y vuelve a graficar con eps=<tu eps> para verificar
min_samples = 2 * X_pcs.shape[1]
plot_k_distancia(X_pcs, k=min_samples)
EPS = 0.8
plot_k_distancia(X_pcs, k=min_samples, eps=EPS)


✍️ **¿Qué observas?** ¿En qué valor de la distancia empieza a dispararse la curva? ¿Qué `eps` elegiste? ¿Qué porcentaje aproximado de clientes queda por encima de tu línea?

- **Dónde se dispara:** alrededor de 0.9 de distancia (el percentil 95 de la k-distancia es 0.89).
- **eps elegido:** 0.8, un poco antes del codo.
- **Por encima de la línea:** aproximadamente 6 % de los clientes (5.9 %).

In [26]:
from sklearn.cluster import DBSCAN

# 1. dbscan_atipicos = DBSCAN(eps=<tu eps>, min_samples=min_samples)
# 2. labels_db = dbscan_atipicos.fit_predict(X_pcs)
# 3. fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
# 4. comparacion.append(fila_db) y muestra fila_db
dbscan_atipicos = DBSCAN(eps=EPS, min_samples=min_samples)
labels_db = dbscan_atipicos.fit_predict(X_pcs)
fila_db = evaluar_clustering('DBSCAN (PCs)', X_pcs, labels_db)
comparacion.append(fila_db)
fila_db


{'modelo': 'DBSCAN (PCs)',
 'n_clusters': 1,
 'pct_ruido': 4.5,
 'silhouette': nan}

In [27]:
# Grafica el resultado de DBSCAN (el ruido sale en gris)
# Pista: plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')
plot_clusters(df_pcs, 'PC1', 'PC2', labels_db, title='DBSCAN sobre PC1 y PC2')


In [28]:
# Esta celda va de regalo: léela con calma y ejecútala
# Sensibilidad: ¿qué pasa con otros valores de eps?
pd.DataFrame([
    evaluar_clustering(f'eps={eps}', X_pcs, DBSCAN(eps=eps, min_samples=dbscan_atipicos.min_samples).fit_predict(X_pcs))
    for eps in [0.4, 0.6, 0.8, 1.0, 1.2]
])

,modelo,n_clusters,pct_ruido,silhouette
0,eps=0.4,4,10.0,0.080
1,eps=0.6,2,7.0,0.344
2,eps=0.8,1,4.5,NaN
3,eps=1.0,2,1.8,0.420
4,eps=1.2,1,1.1,NaN


✍️ **¿Qué observas?** ¿Cuántos clusters encontró DBSCAN y qué porcentaje es ruido? ¿Por qué la silhouette sale `NaN`? ¿Qué te dice esto sobre si los clientes forman grupos separados o un continuo? Según la tabla de sensibilidad, ¿qué tan estable es el porcentaje de ruido?

- **Resultado:** con eps = 0.8 DBSCAN encontró 1 solo cluster y 4.5 % de ruido (20 clientes).
- **Silhouette NaN:** la silhouette necesita al menos 2 clusters, y aquí hay uno solo.
- **Lectura:** los clientes forman un continuo denso, no grupos separados por huecos.
- **Sensibilidad:** el ruido baja gradualmente de 10 % (eps = 0.4) a 1.1 % (eps = 1.2), pero el número de clusters salta (4, 2, 1, 2, 1) sin estabilizarse. No hay una estructura de grupos estable.

### 🔎 ¿Quiénes son los atípicos?

Miramos a los clientes marcados como ruido **en unidades originales** (u.m. al año), ordenados por su gasto total.

In [29]:
# Esta celda va de regalo: léela con calma y ejecútala
df['gasto_total'] = df[gasto].sum(axis=1)

atipicos = df[labels_db == -1].sort_values('gasto_total', ascending=False)
atipicos[gasto + ['gasto_total', 'canal']]

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,canal
85,16117,46197,92780,1026,40827,2944,199891,Retail
47,44466,54259,55571,7782,24171,6465,192714,Retail
181,112151,29627,18148,16745,4948,8550,190169,Horeca
183,36847,43950,20170,36534,239,47943,185683,Horeca
325,32717,16784,13626,60869,1272,5609,130877,Horeca
23,26373,36423,22019,5154,4337,16523,110829,Retail
65,85,20959,45828,36,24231,1423,92562,Retail
109,1406,16729,28986,673,836,3,48633,Retail
289,42786,286,471,1388,32,22,44985,Horeca
357,37,1275,22272,137,6747,110,30578,Retail


✍️ **¿Qué observas?** Mira a los clientes atípicos: ¿todos son grandes compradores? ¿Ves algún patrón distinto entre ellos? ¿Qué tipos de atípicos distingues? ¿Por qué el logaritmo hizo visibles a algunos de ellos?

- **¿Todos son grandes compradores?** No. Los primeros seis tienen gasto total entre 110 mil y 200 mil u.m., pero otros gastan muy poco (904 u.m. el cliente 154, 2,158 el 97 y 2,476 el 98).
- **Tipos de atípicos:** (1) megacompradores de despensa y limpieza, como los clientes 85 y 47 (Retail); (2) megacompradores de una sola categoría, como el 181 en frescos (112 mil), el 183 en delicatessen (48 mil) y el 325 en congelados (61 mil); (3) clientes casi nulos, que compran casi solo una categoría o prácticamente nada.
- **Papel del logaritmo:** hizo visibles a los del tercer tipo. En montos crudos estaban pegados al cero junto con todos los demás. En escala log, pasar de 3 a 500 u.m. es una diferencia enorme y quedan lejos del grupo.

---
# 6️⃣ Perfil de los segmentos en unidades originales (20 min)

Un cluster sólo es útil si se puede **explicar en el idioma del negocio**. Nadie en el área comercial entiende "PC1 = 1.18": hay que regresar a **u.m. al año**. Por eso aplicamos `perfil_clusters` sobre el `df` **original**, sin logaritmo ni escalado.

In [30]:
# Perfil de cada cluster en unidades originales (u.m. al año)
# Pista: perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
perfil_final = perfil_clusters(df, labels_final, gasto + ['gasto_total'])
perfil_final


,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Cluster 0,118,26.8,9770.1,1429.9,1952.9,2292.8,321.3,553.0,16320.0
Cluster 1,106,24.1,12034.2,13077.8,17447.9,3032.8,7272.4,2904.8,55769.8
Cluster 2,72,16.4,3272.4,6618.7,11542.8,591.2,4865.0,658.3,27548.5
Cluster 3,144,32.7,18166.8,3603.1,4080.3,4979.5,755.4,1738.8,33323.9


In [31]:
# Esta celda va de regalo: léela con calma y ejecútala
# Como los gastos son muy sesgados, la mediana describe mejor al cliente típico de cada cluster
medianas = df[gasto + ['gasto_total']].groupby(labels_final).median().round(0)
medianas.index = [f'Cluster {c}' for c in medianas.index]

# ¿Qué porcentaje de la venta total aporta cada cluster?
medianas['pct_de_la_venta'] = (
    100 * df.groupby(labels_final)['gasto_total'].sum() / df['gasto_total'].sum()
).round(1).values
medianas

,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total,pct_de_la_venta
Cluster 0,7814.0,1090.0,1672.0,1428.0,200.0,406.0,14066.0,13.2
Cluster 1,8115.0,9710.0,13662.0,1606.0,5100.0,1852.0,46382.0,40.4
Cluster 2,1492.0,5818.0,10105.0,376.0,4015.0,318.0,26742.0,13.6
Cluster 3,15470.0,3122.0,3444.0,3548.0,542.0,1306.0,28340.0,32.8


✍️ **¿Qué observas?** Para cada cluster: ¿en qué categorías gasta más y en cuáles casi nada? ¿Cuál es el más grande en número de clientes? ¿Cuál aporta más venta? Ponle un **nombre de negocio** a cada segmento.

| Cluster | Nombre del segmento | ¿Qué lo distingue? |
|---|---|---|
| 0 | Horeca pequeño | Gasto bajo en todo (mediana de 14,066 u.m.). Lo más alto es frescos (7,814) y casi no compra detergentes (200). |
| 1 | Tiendas grandes de despensa | Lacteos (9,710), abarrotes (13,662) y detergentes_papel (5,100) muy altos. Aporta 40.4 % de la venta. |
| 2 | Retail pequeño de despensa | Abarrotes (10,105), lacteos (5,818) y detergentes_papel (4,015), con casi nada de frescos (1,492) y congelados (376). |
| 3 | Horeca de perecederos | Frescos (15,470) y congelados (3,548) altos. Es el más numeroso y aporta 32.8 % de la venta. |

(Agrega una fila por cada cluster de tu modelo.)

- **Más grande en clientes:** el cluster 3 (144 clientes, 32.7 %).
- **Más venta:** el cluster 1 aporta 40.4 % de la venta con solo 24.1 % de los clientes.
- **Canal:** el cluster 0 es 100 % Horeca, el 3 es 91.7 % Horeca, el 1 es 83 % Retail y el 2 es 58 % Retail.

In [32]:
# Perfil del ruido de DBSCAN contra el cluster denso
# Pista: perfil_clusters(df, labels_db, gasto + ['gasto_total'])
perfil_clusters(df, labels_db, gasto + ['gasto_total'])


,n,pct,frescos,lacteos,abarrotes,congelados,detergentes_papel,delicatessen,gasto_total
Ruido,20,4.5,16775.5,14424.4,18206.0,6878.4,5984.6,4590.0,66858.8
Cluster 0,420,95.5,11772.9,5385.4,7463.0,2890.7,2733.7,1378.9,31624.6


In [33]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿En qué segmento de K-Means había quedado cada atípico?
pd.crosstab(
    pd.Series(labels_db, name='DBSCAN').map(lambda c: 'Ruido' if c == -1 else 'Denso'),
    pd.Series(labels_final, name='K-Means')
)

K-Means,0,1,2,3
DBSCAN,,,,
Denso,113,100,63,144
Ruido,5,6,9,0


✍️ **¿Qué observas?** ¿El promedio del ruido describe bien a los atípicos? ¿Por qué no? ¿En qué clusters de K-Means habían quedado? ¿Qué pasaría si no los separáramos?

- **¿El promedio describe a los atípicos?** No. El promedio del ruido (gasto total de 66,859 u.m.) corresponde a un cliente grande en todo, pero los atípicos son de tipos muy distintos (megacompradores y casi nulos). El promedio mezcla dos extremos y describe a un cliente que no existe.
- **Dónde habían quedado en K-Means:** 5 en el cluster 0, 6 en el 1, 9 en el 2 y ninguno en el 3.
- **Si no los separáramos:** jalarían los centroides y los perfiles de los segmentos (por ejemplo inflarían los promedios de los clusters 1 y 2) y deformarían la lectura del negocio.

---
# 7️⃣ Validación contra el canal de venta (15 min)

Ahora sí sacamos la variable que guardamos: `canal` (Horeca o Retail). **No la usamos para agrupar**, así que es una prueba honesta: si nuestros segmentos tienen sentido, deberían relacionarse con el tipo de negocio del cliente.

⚠️ En la mayoría de los proyectos reales **no tienes** una etiqueta así. Aquí tenemos la suerte de poder revisar; aun así, no buscamos que los clusters **copien** el canal: buscamos que lo **expliquen** y, si se puede, que digan algo más.

In [34]:
# Tabla cruzada: canal (filas) contra tus clusters (columnas)
# Pista: pd.crosstab(df['canal'], labels_final, rownames=['canal'], colnames=['cluster'], margins=True)
pd.crosstab(df['canal'], labels_final, rownames=['canal'], colnames=['cluster'], margins=True)


cluster,0,1,2,3,All
canal,,,,,
Horeca,118,18,30,132,298
Retail,0,88,42,12,142
All,118,106,72,144,440


In [35]:
# Channel tiene 2 categorías: ajusta también un K-Means con k = 2 para compararlo de tú a tú
# 1. kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
# 2. labels_k2 = kmeans_k2.fit_predict(X_pcs)
# 3. comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
# 4. Tabla cruzada: pd.crosstab(df['canal'], labels_k2, rownames=['canal'], colnames=['cluster'], margins=True)
kmeans_k2 = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_k2 = kmeans_k2.fit_predict(X_pcs)
comparacion.append(evaluar_clustering('K-Means k=2 (PCs)', X_pcs, labels_k2))
pd.crosstab(df['canal'], labels_k2, rownames=['canal'], colnames=['cluster'], margins=True)


cluster,0,1,All
canal,,,
Horeca,54,244,298
Retail,134,8,142
All,188,252,440


In [36]:
# Esta celda va de regalo: léela con calma y ejecútala
# Mismas tablas, pero en % de cada cluster (cada columna suma 100)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    print(f'--- {nombre}: % de cada cluster por canal ---')
    print((100 * pd.crosstab(df['canal'], labels, rownames=['canal'], colnames=['cluster'], normalize='columns')).round(1))

# ¿Cuántos clientes quedan en un cluster cuyo canal mayoritario NO es el suyo?
# (en cada columna, todo lo que no está en la celda más grande)
for nombre, labels in [('K-Means final', labels_final), ('K-Means k=2', labels_k2)]:
    tabla = pd.crosstab(df['canal'], labels)
    print(f'{nombre}: {len(labels) - tabla.max(axis=0).sum()} de {len(labels)} clientes fuera del canal mayoritario de su cluster')

--- K-Means final: % de cada cluster por canal ---
cluster      0     1     2     3
canal                           
Horeca   100.0  17.0  41.7  91.7
Retail     0.0  83.0  58.3   8.3
--- K-Means k=2: % de cada cluster por canal ---
cluster     0     1
canal              
Horeca   28.7  96.8
Retail   71.3   3.2
K-Means final: 60 de 440 clientes fuera del canal mayoritario de su cluster
K-Means k=2: 62 de 440 clientes fuera del canal mayoritario de su cluster


In [37]:
# Esta celda va de regalo: léela con calma y ejecútala
# ¿Y la región? Comparamos la mezcla de regiones de cada cluster con la del total
mezcla_region = 100 * pd.crosstab(df['region'], labels_final, rownames=['region'], colnames=['cluster'], normalize='columns')
mezcla_region['Total'] = 100 * df['region'].value_counts(normalize=True)
mezcla_region.round(1)

cluster,0,1,2,3,Total
region,,,,,
Lisboa,17.8,17.0,13.9,19.4,17.5
Oporto,9.3,11.3,9.7,11.8,10.7
Otra,72.9,71.7,76.4,68.8,71.8


In [38]:
# Dibuja el biplot coloreado por el canal real
# Pista: plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')
plot_biplot(X_pcs, pca_final, gasto, labels=df['canal'], title='Biplot coloreado por canal')


✍️ **¿Qué observas?** ¿Algún cluster es casi puro de un canal? ¿Qué % de cada uno de tus segmentos es Retail? ¿Cuál de los modelos se parece más al canal (cuántos clientes quedan fuera del canal mayoritario de su cluster)? Si tu modelo final se parece menos al canal, ¿significa que está mal? ¿La mezcla de regiones cambia de un cluster a otro? ¿Qué te dice eso sobre organizar a los vendedores por zona? En el biplot, ¿hacia dónde se cargan los Retail y los Horeca?

- **Clusters casi puros:** el cluster 0 es 100 % Horeca y el 3 es 91.7 % Horeca.
- **% de Retail por segmento:** cluster 0 = 0 %, cluster 1 = 83 %, cluster 2 = 58.3 % y cluster 3 = 8.3 %.
- **Modelo más parecido al canal:** son casi iguales. K-Means final deja 60 de 440 clientes fuera del canal mayoritario de su cluster y k = 2 deja 62.
- **¿Está mal si se parece menos?** No. No buscamos copiar el canal sino explicarlo y decir algo más, por ejemplo distinguir Horeca chico de Horeca de perecederos y dos tipos de Retail.
- **Región:** la mezcla casi no cambia entre clusters ('Otra' va de 69 % a 76 %). Organizar vendedores por zona geográfica no ayudaría, porque lo que distingue a los clientes es su canasta.
- **Biplot:** los Retail se cargan hacia la derecha (PC1 promedio de +1.70) y los Horeca hacia la izquierda (−0.81).

---
# 8️⃣ Recomendaciones de negocio y cierre (15 min)

El último paso del flujo es **actuar**. Para cada segmento decide: qué **equipo de venta** lo atiende, qué **surtido** se le ofrece y qué **logística** necesita. Y no olvides a los atípicos.

### 💼 Recomendaciones por segmento

✍️ Completa la tabla con **tus** segmentos (los nombres que elegiste en la Parte 6) y tus números.

| Segmento | Clientes | % de la venta | Equipo de venta | Surtido | Logística |
|---|---|---|---|---|---|
| Horeca pequeño (cluster 0) | 118 | 13.2 % | Equipo de cuentas pequeñas, por teléfono | Catálogo básico de frescos | Rutas consolidadas, entrega semanal |
| Tiendas grandes de despensa (cluster 1) | 106 | 40.4 % | Ejecutivos de cuenta dedicados | Lacteos, abarrotes y detergentes en volumen | Entregas en tarimas completas y descuento por volumen |
| Retail pequeño de despensa (cluster 2) | 72 | 13.6 % | Preventa por teléfono o aplicación | Abarrotes y lacteos | Rutas compartidas, entrega quincenal |
| Horeca de perecederos (cluster 3) | 144 | 32.8 % | Vendedores de campo especializados en restaurantes y hoteles | Frescos, congelados y delicatessen | Entregas frecuentes con cadena de frío |
| Atípicos (DBSCAN) | 20 | — | Gerentes de cuentas clave | Catálogo a la medida | Logística dedicada caso por caso |

(Agrega una fila por cada segmento de tu modelo.)

✍️ ¿Qué harías con los clientes atípicos? ¿Les darías el mismo trato a todos?

Los 20 atípicos (4.5 % de los clientes) no deberían recibir el mismo trato. Los megacompradores merecen un gerente de cuenta dedicado y condiciones a la medida, porque cada uno pesa mucho en la venta. Los casi nulos conviene revisarlos uno por uno: pueden ser clientes dormidos, errores de captura o clientes de una sola categoría, y a cada caso le toca una acción distinta.

### 📊 Tabla comparativa final

In [39]:
# 1. tabla_comparacion = pd.DataFrame(comparacion)
# 2. Muestra tabla_comparacion
tabla_comparacion = pd.DataFrame(comparacion)
tabla_comparacion


,modelo,n_clusters,pct_ruido,silhouette
0,K-Means final (PCs),4,0.0,0.329
1,DBSCAN (PCs),1,4.5,NaN
2,K-Means k=2 (PCs),2,0.0,0.408


✍️ **¿Qué observas?** ¿Cuál modelo gana en cada métrica? ¿Por qué sí podemos comparar estas silhouettes entre sí (a diferencia de la del espacio crudo)? ¿Qué aporta DBSCAN si no tiene silhouette?

- **Quién gana:** k = 2 tiene la silhouette más alta (0.408), seguido de K-Means final (0.329). DBSCAN no tiene silhouette porque solo encontró 1 cluster.
- **Por qué sí se pueden comparar:** las tres silhouettes están calculadas sobre el mismo espacio (`X_pcs`), con la misma escala. La del espacio crudo estaba en otro espacio.
- **Qué aporta DBSCAN:** identifica a los 20 clientes atípicos (4.5 %) que K-Means había escondido dentro de sus segmentos.

## ❓ Respuesta a la pregunta

> **¿Cómo debe organizar el distribuidor a sus clientes mayoristas?**

✍️ Escribe tu respuesta en 4-6 líneas: cuántos segmentos, cómo se llaman, cuántos clientes y qué % de la venta trae cada uno, qué equipo/surtido/logística le toca, qué hacer con los atípicos y si la región o el canal bastarían para organizarlos.

Recomiendo 4 segmentos más una lista aparte de atípicos: Horeca pequeño (118 clientes, 13.2 % de la venta), tiendas grandes de despensa (106 clientes, 40.4 %), Retail pequeño de despensa (72 clientes, 13.6 %) y Horeca de perecederos (144 clientes, 32.8 %). Cada segmento necesita su propio equipo de venta, surtido y logística, como se detalla en la tabla anterior. Los 20 atípicos se atienden con gerentes de cuenta. Ni la región ni el canal bastan por sí solos: la mezcla de regiones es casi igual en todos los segmentos, y Horeca y Retail mezclan clientes grandes con pequeños.

## 📝 Lo que aprendimos

✍️ Escribe las 3 a 5 lecciones más importantes de este caso (y del curso).

1. Los gastos son muy sesgados: aplicar `log1p` antes de escalar evita que unos pocos clientes enormes dominen las distancias.
2. PCA resume seis categorías en dos ejes con significado de negocio (despensa y limpieza contra perecederos).
3. La silhouette no se puede comparar entre espacios distintos (crudo contra log), solo sobre el mismo espacio.
4. K-Means segmenta a todos los clientes y DBSCAN encuentra a los que no encajan: se complementan.
5. Un cluster solo sirve si se traduce a unidades originales y a decisiones: equipo de venta, surtido y logística.

## 🚀 Retos opcionales

1. Repite el análisis con **un componente más** de los que elegiste. ¿Cambian los segmentos?
2. Prueba **K-Means con otro k**. ¿El segmento nuevo es accionable o sólo parte uno existente?
3. Usa **HDBSCAN** (`sklearn.cluster.HDBSCAN`) sobre `X_pcs`. ¿Encuentra los mismos atípicos?
4. Agrupa con **proporciones** en vez de montos: divide cada gasto entre el `gasto_total` del cliente. ¿Aparecen segmentos por *tipo* de canasta sin importar el tamaño?